# Experiment 46 — Canonical SparseWalker credit sanity ladder

This repairs Experiment 45's confounds. The full-control arm calls the existing canonical training path. Event-local and TBPTT-4 use the same default initialization, FullCE, AdamW, BF16, batch size, windows, and 50-epoch LR schedule. The diagnostic run executes 15 epochs by default.


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, sys, shutil, subprocess, torch
from pathlib import Path
REPO='/content/Sparsewalker'
BRANCH='research/active'
if os.path.exists(REPO): shutil.rmtree(REPO)
subprocess.run(['git','clone','-q','-b',BRANCH,'https://github.com/hanialshater/Sparsewalker-.git',REPO],check=True)
for p in [f'{REPO}/src', f'{REPO}/experiments']:
    if p not in sys.path: sys.path.insert(0,p)
HEAD=subprocess.check_output(['git','-C',REPO,'rev-parse','HEAD'],text=True).strip()
import sparsewalker
print('GPU',torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,flush=True)
print('HEAD',HEAD,flush=True)
print('IMPORT_OK',sparsewalker.__file__,flush=True)
assert torch.cuda.is_available(), 'GPU runtime required'


## Run

The first arm is `canonical_full`. If it does not rise clearly above the broken Experiment-45 regime, do not interpret the local arms. The LR schedule is still the canonical 50-epoch schedule, so epoch 15 has **not** decayed as if training were ending.


In [ ]:
import runpy, os, sys
SCRIPT=f'{REPO}/experiments/run_ml1m_canonical_credit_sanity.py'
text=Path(SCRIPT).read_text()
assert 'Experiment 46' in text and 'canonical_full' in text
EPOCHS=15
argv=[SCRIPT,
      '--epochs',str(EPOCHS),
      '--schedule-epochs','50',
      '--batch-size','128',
      '--eval-batch-size','1024',
      '--eval-every','1',
      '--data-dir','/content/drive/MyDrive/sparsewalker_data']
print('RUNNING_IN_PROCESS',' '.join(argv),flush=True)
old_argv=sys.argv[:]; old_cwd=os.getcwd(); sys.argv=argv; os.chdir(REPO)
try:
    runpy.run_path(SCRIPT,run_name='__main__')
finally:
    sys.argv=old_argv; os.chdir(old_cwd)


## Result


In [ ]:
import json, pandas as pd
root=Path('/content/drive/MyDrive/sparsewalker_canonical_credit_sanity/seed42')
sp=root/'summary.json'
if sp.exists():
    summary=json.loads(sp.read_text())
    print('CONTROL_VALID',summary.get('control_valid_min_0.08'))
    display(pd.DataFrame(summary['ranking']))
    print(json.dumps(summary,indent=2))
else:
    print('summary.json not found yet')
